In [1]:
from google.colab import auth



auth.authenticate_user()



print("Authenticated successfully!")


Authenticated successfully!


In [2]:
import pandas as pd



from google.cloud import bigquery

In [3]:
project_id = 'customer-intelligence-db'



client = bigquery.Client(project=project_id)

In [4]:
query = """



SELECT *



FROM raw_retail_data.cleaned_transactions



"""

In [5]:
df = client.query(query).to_dataframe()

In [6]:
display(df.head())

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalSales
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00+00:00,2.55,17850,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00+00:00,3.39,17850,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00+00:00,2.75,17850,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00+00:00,3.39,17850,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00+00:00,3.39,17850,United Kingdom,20.34


In [7]:
# 1. Converting InvoiceDate to a proper time format so that we can calculate days
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# 2. This is to find the most recent date in the dataset to act as our "today"
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

# 3. Grouping the millions of rows by CustomerID to calculate R, F, and M
customer_data = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days, # Recency: Days since last purchase
    'InvoiceNo': 'nunique',                                  # Frequency: Count of unique orders
    'TotalSales': 'sum'                                      # Monetary: Total dollars spent
}).reset_index()

# 4. Renaming the Columns for clarity
customer_data.rename(columns={
    'InvoiceDate': 'Recency',
    'InvoiceNo': 'Frequency',
    'TotalSales': 'Monetary'
}, inplace=True)

# 5. Define Churn: Here, if a customer hasn't bought anything in 90 days, they would be tagged as 1 (Churned). Otherwise, 0 (Retained).
customer_data['Churn'] = customer_data['Recency'].apply(lambda x: 1 if x > 90 else 0)

# 6. This is just to display clean customer profiles
display(customer_data.head())

,CustomerID,Recency,Frequency,Monetary,Churn
0,12346,326,1,77183.60,1
1,12347,2,7,4310.00,0
2,12348,75,4,1797.24,0
3,12349,19,1,1757.55,0
4,12350,310,1,334.40,1


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# 1. Define our Features (X) and our Target (y)
# X contains the clues (Frequency and Money). y contains the answer (Churn).
X = customer_data[['Frequency', 'Monetary']]
y = customer_data['Churn']

# 2. Split the data: 80% to train the model, 20% to test it on data it has never seen
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Creating the Machine Learning model
print("Training the AI model...")
model = RandomForestClassifier(random_state=42)

# 4. Training the model using 80% dataset
model.fit(X_train, y_train)
print("Training complete!\n")

# 5. Here, I am asking the model to take a "test" using the remaining 20% of data
predictions = model.predict(X_test)

# 6. Here, I am grading the test to see how well it learned the patterns
accuracy = accuracy_score(y_test, predictions)
print(f"Model Accuracy: {accuracy * 100:.2f}%\n")
print("Detailed Performance Report:")
print(classification_report(y_test, predictions))

Training the AI model...
Training complete!

Model Accuracy: 63.94%

Detailed Performance Report:
              precision    recall  f1-score   support

           0       0.70      0.77      0.73       561
           1       0.49      0.41      0.44       307

    accuracy                           0.64       868
   macro avg       0.60      0.59      0.59       868
weighted avg       0.63      0.64      0.63       868



In [9]:
all_predictions = model.predict(X)

In [10]:
customer_data['Predicted_Churn_Risk'] = all_predictions

In [11]:
display(customer_data[['CustomerID', 'Predicted_Churn_Risk']].head())

,CustomerID,Predicted_Churn_Risk
0,12346,1
1,12347,0
2,12348,0
3,12349,0
4,12350,1


In [12]:
table_id = f"{project_id}.raw_retail_data.customer_churn_predictions"

In [13]:
print("Uploading predictions to BigQuery...")
customer_data.to_gbq(destination_table=table_id, project_id=project_id, if_exists='replace')
print("Upload successful! The data is now in your cloud warehouse.")

Uploading predictions to BigQuery...


/tmp/ipykernel_1021/2561185972.py:2: FutureWarning: to_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.to_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.to_gbq
  customer_data.to_gbq(destination_table=table_id, project_id=project_id, if_exists='replace')
100%|██████████| 1/1 [00:00<00:00, 11781.75it/s]

Upload successful! The data is now in your cloud warehouse.
